# IssueFix-RL — OPSD Training on Kaggle 2× T4

This notebook runs a **single process** that places the trainable student on GPU 0 and the frozen privileged-context teacher on GPU 1. Do not wrap the training command with `accelerate launch`.

Before running: enable the **GPU T4 ×2** accelerator, enable Internet, attach the JSONL training dataset, and optionally add a Kaggle secret named `WANDB_API_KEY`.

In [ ]:
# EDIT THESE VALUES
from datetime import datetime, timezone

REPO_URL = "https://github.com/ramprasathk07/IssueFix-RL.git"
DATA_PATH = "/kaggle/input/datasets/ramprasathk07/thinking-traces/opencode_sft_filtered_sl3072_10000.jsonl"
FINETUNING = "lora"  # choose: "lora" or "full"
RESUME_CHECKPOINT = None  # e.g. /kaggle/input/my-checkpoint/checkpoint-epoch1-step100
WANDB_PROJECT = "issuefix_opsd"
RUN_NAME = f"qwen0.5_opsd_{FINETUNING}_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M')}"

assert FINETUNING in {"lora", "full"}


In [ ]:
import os, subprocess

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
REPO_DIR = "/kaggle/working/IssueFix-RL"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
print("Repository:", os.getcwd())


In [ ]:
# Kaggle already provides CUDA PyTorch. Install repository dependencies.
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)


In [ ]:
# Fail early unless Kaggle really assigned two CUDA GPUs.
import torch

assert torch.cuda.is_available(), "CUDA is unavailable. Enable a Kaggle GPU accelerator."
assert torch.cuda.device_count() >= 2, (
    f"OPSD needs two GPUs, but Kaggle exposed {torch.cuda.device_count()}. "
    "Select the GPU T4 x2 accelerator."
)
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"cuda:{index}: {props.name}, {props.total_memory / 2**30:.1f} GiB")


In [ ]:
# Optional Weights & Biases authentication.
try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = key
    print("WANDB_API_KEY loaded")
except Exception:
    os.environ["WANDB_DISABLED"] = "true"
    print("No WANDB_API_KEY; WandB disabled")


In [ ]:
# Validate the attached dataset before allocating model memory.
import json
from pathlib import Path

data_file = Path(DATA_PATH)
assert data_file.is_file(), f"Dataset not found: {data_file}"
with data_file.open(encoding="utf-8") as handle:
    rows = [json.loads(line) for line in handle if line.strip()]
assert rows, "The dataset is empty"
sample = rows[0]
assert sample.get("problem") or sample.get("prompt"), "Missing problem/prompt field"
assert sample.get("solution") or sample.get("response"), "Missing solution/response field"
print(f"Rows: {len(rows):,}; keys: {sorted(sample)}")
del rows


In [ ]:
# Derive a session config. The checked-in preset remains unchanged.
import yaml

with open("configs/opsd.yaml", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)
cfg["model_params"]["use_lora"] = FINETUNING == "lora"
cfg["model_params"]["load_in_4bit"] = False
cfg["training_params"]["learning_rate"] = 2e-4 if FINETUNING == "lora" else 2e-5
cfg["training_params"]["wandb_project"] = WANDB_PROJECT
cfg["training_params"]["wandb_run_name"] = RUN_NAME
cfg["training_params"]["output_dir"] = f"/kaggle/working/outputs/opsd_{FINETUNING}"
cfg["opsd_params"]["student_device"] = "cuda:0"
cfg["opsd_params"]["teacher_device"] = "cuda:1"
runtime_config = Path("/kaggle/working/opsd-runtime.yaml")
with runtime_config.open("w", encoding="utf-8") as handle:
    yaml.safe_dump(cfg, handle, sort_keys=False)
print(yaml.safe_dump({
    "finetuning": FINETUNING,
    "model": cfg["model_params"]["base_model"],
    "learning_rate": cfg["training_params"]["learning_rate"],
    "student": cfg["opsd_params"]["student_device"],
    "teacher": cfg["opsd_params"]["teacher_device"],
    "completion_tokens": cfg["opsd_params"]["max_completion_length"],
    "output_dir": cfg["training_params"]["output_dir"],
}, sort_keys=False))


In [ ]:
# TRAINING: intentionally one process controlling both GPUs.
command = [
    sys.executable, "train.py",
    "--method", "opsd",
    "--finetuning", FINETUNING,
    "--config", str(runtime_config),
    "--data", str(data_file),
    "--wandb_project", WANDB_PROJECT,
    "--wandb_run_name", RUN_NAME,
]
if RESUME_CHECKPOINT:
    command.extend(["--resume", RESUME_CHECKPOINT])
print("Launching:", " ".join(command))
subprocess.run(command, check=True, env=os.environ.copy())


In [ ]:
# List and archive outputs so they appear in the Kaggle notebook Output tab.
import shutil

out_dir = Path(cfg["training_params"]["output_dir"])
checkpoints = sorted(out_dir.glob("checkpoint-*"), key=lambda p: p.stat().st_mtime)
assert checkpoints, f"No checkpoints found under {out_dir}"
for checkpoint in checkpoints:
    size_mb = sum(p.stat().st_size for p in checkpoint.rglob("*") if p.is_file()) / 2**20
    print(f"{checkpoint.name}: {size_mb:.1f} MiB")
latest = checkpoints[-1]
archive_base = Path("/kaggle/working") / f"{RUN_NAME}-{latest.name}"
archive = shutil.make_archive(str(archive_base), "zip", root_dir=latest)
print(f"Archived latest checkpoint: {archive}")


## Notes

- LoRA is the recommended T4 setting and uses learning rate `2e-4`.
- Full fine-tuning uses `2e-5`, an 8-bit optimizer, fp16, and gradient checkpointing.
- The teacher stays frozen and 4-bit in both modes.
- To resume in a later Kaggle session, attach the checkpoint directory as a dataset and set `RESUME_CHECKPOINT` in the first code cell.